# California EV Charger Needs Assessment by Census Tract

## Throughput-Based Model

**Research Question:** If all gasoline cars in California are replaced by EVs, how many public EV charging ports are needed per census tract?

### Model Logic
1. **Gas pump throughput**: 1 pump serves ~12 cars/hr (5 min/car)
2. **EV charger throughput**:
   - DCFC (fast charger): ~2.5 cars/hr (25 min/car)
   - Level 2: ~0.33 cars/hr (3 hrs/car)
3. **Throughput ratio**: 1 gas pump ≈ 12.2 EV ports (blended 30% DCFC + 70% L2)
4. **Home charging**: ~80% of EV charging happens at home
5. **Public need**: Only 20% of charging demand requires public infrastructure
6. **Formula**: `needed_ports = gas_nozzles × 12.2 × 0.20`

### Data Sources
- Census tract shapefile: US Census Bureau TIGER/Line 2024 (FIPS 06 = California)
- County-level charger data: CEC EV Chargers Dashboard (2025)
- County gas nozzle estimates: CEC / AFDC
- Throughput benchmarks: DOE, industry studies

### Allocation Method
County-level data → tract-level via inverse-sqrt(area) weighting (smaller tracts = more urban = more chargers)


## 1. Setup & Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
print("Libraries loaded successfully.")


## 2. Load California Census Tract Shapefile

TIGER/Line 2024, FIPS code 06 = California. Contains 9,129 census tracts.


In [ ]:
# ── Load shapefile ──
# Update this path to wherever your shapefile is located
SHAPEFILE_PATH = "tl_2024_06_tract.shp"

gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"Tracts loaded: {len(gdf)}")
print(f"CRS: {gdf.crs}")
print(f"Counties: {gdf['COUNTYFP'].nunique()}")

# Compute area in square miles (project to California Albers)
gdf_proj = gdf.to_crs(epsg=3310)
gdf['area_sqmi'] = gdf_proj.geometry.area / 2.59e6  # m² → sq mi
gdf['INTPTLAT'] = gdf['INTPTLAT'].astype(float)
gdf['INTPTLON'] = gdf['INTPTLON'].astype(float)
gdf.loc[gdf['area_sqmi'] <= 0, 'area_sqmi'] = 0.001

gdf[['GEOID', 'COUNTYFP', 'NAMELSAD', 'area_sqmi']].head(10)


## 3. County-Level Data

County-level data for:
- **Gas nozzles** (estimated from CEC / AFDC)
- **Total registered vehicles** (CA DMV Jan 2026: ~36.2M statewide)
- **Existing public EVSE ports** (CEC Dashboard 2025: ~106K public ports)


In [ ]:
# ── County-level data: (name, gas_nozzles, total_vehicles, existing_public_evse) ──
county_data = {
    '001': ('Alameda',         1800, 1156000, 5450),
    '003': ('Alpine',            10,    1200,    7),
    '005': ('Amador',           100,   30000,   55),
    '007': ('Butte',            500,  150000,  335),
    '009': ('Calaveras',        120,   38000,   45),
    '011': ('Colusa',            60,   15000,   20),
    '013': ('Contra Costa',    1300,  810000, 3680),
    '015': ('Del Norte',         60,   18000,   33),
    '017': ('El Dorado',        350,  145000,  345),
    '019': ('Fresno',          1800,  620000, 1130),
    '021': ('Glenn',             70,   20000,   25),
    '023': ('Humboldt',         300,   95000,  220),
    '025': ('Imperial',         350,  100000,  105),
    '027': ('Inyo',              50,   14000,   50),
    '029': ('Kern',            1600,  530000,  610),
    '031': ('Kings',            250,   85000,   90),
    '033': ('Lake',             130,   42000,   65),
    '035': ('Lassen',            70,   22000,   20),
    '037': ('Los Angeles',    14000, 7200000, 31800),
    '039': ('Madera',           300,  100000,  130),
    '041': ('Marin',            300,  210000, 1380),
    '043': ('Mariposa',          50,   14000,   40),
    '045': ('Mendocino',        180,   62000,  165),
    '047': ('Merced',           450,  160000,  185),
    '049': ('Modoc',             25,    7000,   11),
    '051': ('Mono',              35,   10000,   42),
    '053': ('Monterey',         750,  280000,  770),
    '055': ('Napa',             180,  105000,  520),
    '057': ('Nevada',           160,   78000,  305),
    '059': ('Orange',          4500, 2400000, 12000),
    '061': ('Placer',           550,  310000,  950),
    '063': ('Plumas',            55,   17000,   33),
    '065': ('Riverside',       3800, 1500000, 3750),
    '067': ('Sacramento',      2800, 1050000, 3280),
    '069': ('San Benito',       100,   42000,  125),
    '071': ('San Bernardino',  3500, 1400000, 2820),
    '073': ('San Diego',       4800, 2300000, 9800),
    '075': ('San Francisco',    600,  520000, 3850),
    '077': ('San Joaquin',     1200,  480000,  710),
    '079': ('San Luis Obispo',  500,  210000,  710),
    '081': ('San Mateo',        800,  560000, 3420),
    '083': ('Santa Barbara',    650,  300000,  930),
    '085': ('Santa Clara',     2200, 1350000, 8600),
    '087': ('Santa Cruz',       400,  190000,  750),
    '089': ('Shasta',           400,  130000,  225),
    '091': ('Sierra',            12,    2800,    7),
    '093': ('Siskiyou',         120,   35000,   55),
    '095': ('Solano',           600,  310000, 1050),
    '097': ('Sonoma',           650,  370000, 1600),
    '099': ('Stanislaus',       950,  350000,  475),
    '101': ('Sutter',           180,   65000,  125),
    '103': ('Tehama',           130,   45000,   45),
    '105': ('Trinity',           35,   10000,   17),
    '107': ('Tulare',           750,  280000,  335),
    '109': ('Tuolumne',         130,   42000,   80),
    '111': ('Ventura',         1300,  620000, 2080),
    '113': ('Yolo',             300,  140000,  580),
    '115': ('Yuba',             130,   50000,   75),
}

county_df = pd.DataFrame([
    {'COUNTYFP': k, 'county_name': v[0], 'gas_nozzles': v[1],
     'total_vehicles': v[2], 'existing_evse': v[3]}
    for k, v in county_data.items()
])

print(f"Counties: {len(county_df)}")
print(f"Total gas nozzles:    {county_df['gas_nozzles'].sum():>10,}")
print(f"Total vehicles:       {county_df['total_vehicles'].sum():>10,}")
print(f"Total existing EVSE:  {county_df['existing_evse'].sum():>10,}")

county_df.sort_values('existing_evse', ascending=False).head(10)


## 4. Throughput Model — How Many EV Ports to Replace Gas Pumps?

### Key parameters:
| Fuel Type | Throughput | Time per Car |
|-----------|-----------|-------------|
| Gas pump | 12 cars/hr | ~5 min |
| DCFC (fast charger) | 2.5 cars/hr | ~25 min |
| Level 2 | 0.33 cars/hr | ~3 hrs |

### Blended EV mix: 30% DCFC + 70% L2
→ **1 gas pump = 12.2 EV charger ports** in throughput equivalence

### Home charging discount: 80%
→ Only **20%** of charging demand needs public infrastructure

### Formula:
```
needed_public_ports = gas_nozzles × 12.2 × 0.20
```


In [ ]:
# ── Throughput model parameters ──
GAS_PUMP_CARS_PER_HR  = 12      # 1 pump: 5 min/car → 12 cars/hr
DCFC_CARS_PER_HR      = 2.5     # 1 DCFC: ~25 min/car
L2_CARS_PER_HR        = 0.33    # 1 L2: ~3 hrs/car
HOME_CHARGE_PCT       = 0.80    # 80% charge at home
PUBLIC_CHARGE_PCT     = 0.20    # 20% needs public infra
DCFC_SHARE            = 0.30    # 30% of public ports = DCFC
L2_SHARE              = 0.70    # 70% of public ports = L2

# Blended throughput
blended_ev_cars_per_hr = DCFC_SHARE * DCFC_CARS_PER_HR + L2_SHARE * L2_CARS_PER_HR
throughput_ratio = GAS_PUMP_CARS_PER_HR / blended_ev_cars_per_hr

print(f"Blended EV throughput:  {blended_ev_cars_per_hr:.2f} cars/hr")
print(f"Throughput ratio:       1 gas pump = {throughput_ratio:.1f} EV ports")
print(f"With 20% public need:   1 gas pump → {throughput_ratio * PUBLIC_CHARGE_PCT:.1f} public EV ports")

# Apply to county data
county_df['needed_ports'] = (county_df['gas_nozzles'] * throughput_ratio * PUBLIC_CHARGE_PCT).round().astype(int)
county_df['needed_dcfc']  = (county_df['needed_ports'] * DCFC_SHARE).round().astype(int)
county_df['needed_l2']    = (county_df['needed_ports'] * L2_SHARE).round().astype(int)
county_df['gap']          = (county_df['needed_ports'] - county_df['existing_evse']).clip(lower=0)
county_df['coverage_pct'] = (county_df['existing_evse'] / county_df['needed_ports'].replace(0, 1) * 100).round(1)

total_needed   = county_df['needed_ports'].sum()
total_existing = county_df['existing_evse'].sum()
total_gap      = county_df['gap'].sum()

print(f"\n{'='*55}")
print(f"STATEWIDE SUMMARY")
print(f"{'='*55}")
print(f"  Gas nozzles statewide:      {county_df['gas_nozzles'].sum():>10,}")
print(f"  Needed public EV ports:     {total_needed:>10,}")
print(f"    DCFC:                     {county_df['needed_dcfc'].sum():>10,}")
print(f"    Level 2:                  {county_df['needed_l2'].sum():>10,}")
print(f"  Existing public EVSE:       {total_existing:>10,}")
print(f"  GAP (still needed):         {total_gap:>10,}")
print(f"  Current coverage:           {total_existing/total_needed*100:>9.1f}%")


## 5. Allocate County Data → Census Tracts

Within each county, chargers are allocated to tracts using **inverse-sqrt(area) weighting**:
- Smaller tracts = more urban = higher population density = more chargers
- Larger tracts = rural = fewer chargers

This is an approximation. For exact results, use AFDC station lat/lon data with spatial join.


In [ ]:
# ── Merge county data to tracts ──
gdf = gdf.merge(county_df, on='COUNTYFP', how='left')

# Inverse-sqrt(area) weighting
gdf['w'] = 1.0 / np.sqrt(gdf['area_sqmi'].clip(lower=0.001))
cw = gdf.groupby('COUNTYFP')['w'].transform('sum')
gdf['wn'] = gdf['w'] / cw

# Allocate each metric
for col in ['gas_nozzles', 'total_vehicles', 'existing_evse',
            'needed_ports', 'needed_dcfc', 'needed_l2', 'gap']:
    gdf[f't_{col}'] = (gdf['wn'] * gdf[col]).round(0).astype(int)

# Derived metrics
gdf['t_coverage_pct'] = np.where(
    gdf['t_needed_ports'] > 0,
    (gdf['t_existing_evse'] / gdf['t_needed_ports'] * 100).round(1),
    100.0
)
gdf['needed_per_sqmi'] = (gdf['t_needed_ports'] / gdf['area_sqmi']).round(1)
gdf['gap_per_sqmi']    = (gdf['t_gap'] / gdf['area_sqmi']).round(1)

print(f"Tract-level allocation complete.")
print(f"\n{'Metric':<40} {'Value':>12}")
print(f"{'-'*54}")
print(f"{'Mean needed ports/tract':<40} {gdf['t_needed_ports'].mean():>12.1f}")
print(f"{'Mean existing EVSE/tract':<40} {gdf['t_existing_evse'].mean():>12.1f}")
print(f"{'Mean gap/tract':<40} {gdf['t_gap'].mean():>12.1f}")
print(f"{'Mean coverage %':<40} {gdf['t_coverage_pct'].mean():>11.1f}%")
print(f"{'Tracts fully covered (>=100%)':<40} {(gdf['t_coverage_pct']>=100).sum():>12,}")
print(f"{'Tracts <50% covered':<40} {(gdf['t_coverage_pct']<50).sum():>12,}")
print(f"{'Tracts with 0 EVSE':<40} {(gdf['t_existing_evse']==0).sum():>12,}")


## 6. Top Tracts — Largest Gaps & Highest Coverage

In [ ]:
# ── Top 20 tracts by GAP (most underserved) ──
print("TOP 20 TRACTS BY CHARGER DEFICIT:")
print("=" * 80)
top_gap = gdf.nlargest(20, 't_gap')[
    ['GEOID', 'county_name', 'NAMELSAD', 't_existing_evse', 't_needed_ports', 't_gap', 't_coverage_pct']
].reset_index(drop=True)
top_gap.index += 1
top_gap.columns = ['GEOID', 'County', 'Tract', 'Existing', 'Needed', 'Gap', 'Coverage%']
display(top_gap)


In [ ]:
# ── County-level summary (sorted by gap) ──
print("\nCOUNTY SUMMARY (Top 15 by gap):")
print("=" * 80)
cs = county_df.nlargest(15, 'gap')[
    ['county_name', 'gas_nozzles', 'needed_ports', 'existing_evse', 'gap', 'coverage_pct']
]
cs.index = range(1, len(cs)+1)
cs.columns = ['County', 'Gas Nozzles', 'Needed Ports', 'Existing EVSE', 'Gap', 'Coverage%']
display(cs)


## 7. Visualization — Choropleth Maps

In [ ]:
def make_viridis_map(ax, gdf, column, title, bins, labels):
    """Create a viridis choropleth matching EVSE standard style."""
    colors = ['#440154','#46327e','#365c8d','#277f8e','#1fa187','#4ac16d','#9fda3a','#fde725']
    n = len(bins) - 1
    cmap = ListedColormap(colors[:n])
    norm = BoundaryNorm(bins, cmap.N)

    gdf.plot(column=column, ax=ax, cmap=cmap, norm=norm,
             edgecolor='#333333', linewidth=0.08)

    legend_elements = [
        Patch(facecolor=colors[i], edgecolor='gray', linewidth=0.5, label=labels[i])
        for i in range(n)
    ]
    ax.legend(handles=legend_elements, title='Ports by tract',
              loc='upper right', fontsize=7, title_fontsize=8,
              frameon=True, fancybox=False, edgecolor='gray', framealpha=0.95)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlim(-124.5, -114)
    ax.set_ylim(32.5, 42.1)
    ax.set_axis_off()

print("Map helper function defined.")


### Map A: Existing Public EV Charger Ports

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))
bins1   = [0, 0.5, 2, 5, 10, 25, 50, 100, 600]
labels1 = ['0', '0 – 2', '2 – 5', '5 – 10', '10 – 25', '25 – 50', '50 – 100', '100 – 518']
make_viridis_map(ax, gdf, 't_existing_evse',
                 'Existing Public EV Chargers Joined to California Census Tracts',
                 bins1, labels1)
plt.tight_layout()
plt.show()


### Map B: Needed Public EV Charger Ports (Full EV Substitution)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))
bins2   = [0, 0.5, 5, 10, 25, 50, 100, 200, 1000]
labels2 = ['0', '0 – 5', '5 – 10', '10 – 25', '25 – 50', '50 – 100', '100 – 200', '200+']
make_viridis_map(ax, gdf, 't_needed_ports',
                 f'Needed Public EV Charger Ports (Full EV Substitution)\n'
                 f'1 gas pump = {throughput_ratio:.0f} EV ports × 20% public need',
                 bins2, labels2)
plt.tight_layout()
plt.show()


### Map C: Charger Gap (Needed − Existing)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

bins3   = [0, 0.5, 2, 5, 10, 25, 50, 100, 800]
labels3 = ['0 (met)', '0 – 2', '2 – 5', '5 – 10', '10 – 25', '25 – 50', '50 – 100', '100+']
colors_gap = ['#1a9850','#91cf60','#d9ef8b','#fee08b','#fdae61','#f46d43','#d73027','#a50026']
cmap_gap = ListedColormap(colors_gap)
norm_gap = BoundaryNorm(bins3, cmap_gap.N)

gdf.plot(column='t_gap', ax=ax, cmap=cmap_gap, norm=norm_gap,
         edgecolor='#333333', linewidth=0.08)

legend_gap = [Patch(facecolor=colors_gap[i], edgecolor='gray', linewidth=0.5,
                    label=labels3[i]) for i in range(len(labels3))]
ax.legend(handles=legend_gap, title='Port deficit by tract',
          loc='upper right', fontsize=7, title_fontsize=8,
          frameon=True, fancybox=False, edgecolor='gray', framealpha=0.95)
ax.set_title('EV Charger Gap: Additional Ports Needed per Tract\n'
             '(Green = need met, Red = large deficit)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlim(-124.5, -114); ax.set_ylim(32.5, 42.1); ax.set_axis_off()
plt.tight_layout()
plt.show()


### Map D: Coverage % (Existing / Needed × 100)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

bins4   = [0, 10, 25, 50, 75, 100, 150, 300, 5000]
labels4 = ['0–10%', '10–25%', '25–50%', '50–75%', '75–100%', '100–150%', '150–300%', '300%+']
colors_cov = ['#a50026','#d73027','#f46d43','#fdae61','#fee08b','#d9ef8b','#91cf60','#1a9850']
cmap_cov = ListedColormap(colors_cov)
norm_cov = BoundaryNorm(bins4, cmap_cov.N)

gdf.plot(column='t_coverage_pct', ax=ax, cmap=cmap_cov, norm=norm_cov,
         edgecolor='#333333', linewidth=0.08)

legend_cov = [Patch(facecolor=colors_cov[i], edgecolor='gray', linewidth=0.5,
                    label=labels4[i]) for i in range(len(labels4))]
ax.legend(handles=legend_cov, title='Coverage %',
          loc='upper right', fontsize=7, title_fontsize=8,
          frameon=True, fancybox=False, edgecolor='gray', framealpha=0.95)
ax.set_title('EV Charger Coverage: Existing / Needed × 100%\n'
             '(Green = sufficient, Red = underserved)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlim(-124.5, -114); ax.set_ylim(32.5, 42.1); ax.set_axis_off()
plt.tight_layout()
plt.show()


### Combined 4-Panel Map

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 28))

# A. Existing
make_viridis_map(axes[0,0], gdf, 't_existing_evse', 'A. Existing Public EVSE Ports', bins1, labels1)

# B. Needed
make_viridis_map(axes[0,1], gdf, 't_needed_ports',
                 f'B. Needed Ports (1 pump = {throughput_ratio:.0f} EV ports × 20%)', bins2, labels2)

# C. Gap
gdf.plot(column='t_gap', ax=axes[1,0], cmap=cmap_gap, norm=norm_gap,
         edgecolor='#333333', linewidth=0.08)
lg = [Patch(facecolor=colors_gap[i], edgecolor='gray', linewidth=0.5,
            label=labels3[i]) for i in range(len(labels3))]
axes[1,0].legend(handles=lg, title='Port deficit', loc='upper right',
                 fontsize=7, title_fontsize=8, frameon=True, edgecolor='gray', framealpha=0.95)
axes[1,0].set_title('C. Gap (Needed − Existing)', fontsize=12, fontweight='bold', pad=10)
axes[1,0].set_xlim(-124.5, -114); axes[1,0].set_ylim(32.5, 42.1); axes[1,0].set_axis_off()

# D. Coverage
gdf.plot(column='t_coverage_pct', ax=axes[1,1], cmap=cmap_cov, norm=norm_cov,
         edgecolor='#333333', linewidth=0.08)
lc = [Patch(facecolor=colors_cov[i], edgecolor='gray', linewidth=0.5,
            label=labels4[i]) for i in range(len(labels4))]
axes[1,1].legend(handles=lc, title='Coverage %', loc='upper right',
                 fontsize=7, title_fontsize=8, frameon=True, edgecolor='gray', framealpha=0.95)
axes[1,1].set_title('D. Coverage % (Green=sufficient, Red=underserved)',
                    fontsize=12, fontweight='bold', pad=10)
axes[1,1].set_xlim(-124.5, -114); axes[1,1].set_ylim(32.5, 42.1); axes[1,1].set_axis_off()

fig.suptitle('California EV Charger Needs Assessment — Throughput Model\n'
             f'Gas pump: {GAS_PUMP_CARS_PER_HR} cars/hr | DCFC: {DCFC_CARS_PER_HR} cars/hr | '
             f'L2: {L2_CARS_PER_HR} cars/hr | Home charging: {HOME_CHARGE_PCT*100:.0f}% | '
             f'Public need: {PUBLIC_CHARGE_PCT*100:.0f}%',
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


### Zoomed Maps: LA Metro & Bay Area

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 20))

# LA - Existing
make_viridis_map(axes[0,0], gdf, 't_existing_evse', 'LA Metro — Existing EVSE', bins1, labels1)
axes[0,0].set_xlim(-118.8, -117.6); axes[0,0].set_ylim(33.65, 34.35)

# LA - Gap
gdf.plot(column='t_gap', ax=axes[0,1], cmap=cmap_gap, norm=norm_gap,
         edgecolor='gray', linewidth=0.1)
lg = [Patch(facecolor=colors_gap[i], edgecolor='gray', linewidth=0.5,
            label=labels3[i]) for i in range(len(labels3))]
axes[0,1].legend(handles=lg, title='Port deficit', loc='upper right', fontsize=7, title_fontsize=8,
                 frameon=True, edgecolor='gray', framealpha=0.95)
axes[0,1].set_title('LA Metro — Charger Gap', fontsize=12, fontweight='bold', pad=10)
axes[0,1].set_xlim(-118.8, -117.6); axes[0,1].set_ylim(33.65, 34.35); axes[0,1].set_axis_off()

# Bay Area - Existing
make_viridis_map(axes[1,0], gdf, 't_existing_evse', 'Bay Area — Existing EVSE', bins1, labels1)
axes[1,0].set_xlim(-122.6, -121.7); axes[1,0].set_ylim(37.2, 37.9)

# Bay Area - Gap
gdf.plot(column='t_gap', ax=axes[1,1], cmap=cmap_gap, norm=norm_gap,
         edgecolor='gray', linewidth=0.1)
axes[1,1].legend(handles=lg, title='Port deficit', loc='upper right', fontsize=7, title_fontsize=8,
                 frameon=True, edgecolor='gray', framealpha=0.95)
axes[1,1].set_title('Bay Area — Charger Gap', fontsize=12, fontweight='bold', pad=10)
axes[1,1].set_xlim(-122.6, -121.7); axes[1,1].set_ylim(37.2, 37.9); axes[1,1].set_axis_off()

fig.suptitle('EV Charger Assessment — LA Metro & Bay Area Zoom', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


## 8. Export Results

In [ ]:
# ── Export tract-level results to CSV ──
export_cols = [
    'GEOID', 'COUNTYFP', 'county_name', 'NAMELSAD',
    'INTPTLAT', 'INTPTLON', 'area_sqmi',
    't_gas_nozzles', 't_total_vehicles',
    't_existing_evse', 't_needed_ports', 't_needed_dcfc', 't_needed_l2',
    't_gap', 't_coverage_pct',
    'needed_per_sqmi', 'gap_per_sqmi'
]
export_df = gdf[export_cols].sort_values('t_gap', ascending=False)
export_df.to_csv('ca_ev_charger_needs_by_tract.csv', index=False)
print(f"Exported: ca_ev_charger_needs_by_tract.csv ({len(export_df)} rows)")

export_df.head(20)


## Summary

### Key Findings:
| Metric | Value |
|--------|-------|
| Gas nozzles statewide | ~57,500 |
| Throughput ratio | 1 gas pump = 12.2 EV ports |
| Public charging need | 20% (80% home charging) |
| **Total public EV ports needed** | **~140,700** |
| Existing public EVSE | ~106,100 |
| **Gap (still needed)** | **~44,900 ports** |
| **Current coverage** | **75.4%** |

### Most Underserved Counties:
1. San Bernardino (32.9% coverage, gap = 5,743)
2. Riverside (40.3%, gap = 5,547)
3. Sacramento (47.9%, gap = 3,570)
4. Kern (15.6%, gap = 3,304)
5. Fresno (25.7%, gap = 3,274)

### Best Covered Counties:
- San Francisco, Marin, San Mateo, Santa Clara — all >100% coverage
- Bay Area leads in EV infrastructure density

### Caveats:
- County → tract allocation uses area-based proxy, not exact station locations
- For precise results, use AFDC station lat/lon with spatial join to tracts
- Home charging % varies by housing type (single-family vs MUD)
- Throughput assumes average utilization; peak hours need ~2x capacity
